# Masar Day 3 — Executed learner evidence

Extracted from the learner consolidated Colab notebook with the original executed code cells and retained outputs.


In [153]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_mtpvm923


In [154]:
from masar.delta_lab import run_transactions_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_transactions_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04a_transactions', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({k: result[k] for k in ('before','after_correction','past_version_read')}, indent=2, default=str))
finally:
    spark.stop()

{
  "scope": "DAY03_TRANSACTIONS_ENGINE",
  "checks": {
    "native_correction_matches_source_expectation": true,
    "business_rows_stay_75": true,
    "replay_and_stale_delivery_preserve_values": true,
    "same_revision_conflict_rejected": true,
    "actual_prior_version_read": true,
    "mixed_valid_invalid_batch_rejected_atomically": true
  }
}
{
  "before": {
    "rows": 75,
    "business_digest": "0d16e2795620ae0c0f54d3fcd52a5b13fb2c2a47cd4d0c42c8ddb195f19bc6e0",
    "version": 1
  },
  "after_correction": {
    "rows": 75,
    "business_digest": "1321d375742d806a1f0fef82be9e2862af04f5d18f3a976792a6b1d9aa1b4383",
    "version": 2
  },
  "past_version_read": 1
}


In [155]:
from masar.delta_lab import run_maintenance_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_maintenance_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04b_maintenance', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({'recovery': result['recovery'], 'vacuum': result['vacuum']}, indent=2, default=str))
finally:
    spark.stop()

{
  "scope": "DAY03_MAINTENANCE_ENGINE",
  "checks": {
    "unexpected_column_rejected": true,
    "approved_evolution_preserves_business_values": true,
    "compaction_preserves_values": true,
    "delete_affects_copy_only": true,
    "restore_creates_new_commit": true,
    "vacuum_is_non_destructive_dry_run": true,
    "trusted_silver_unchanged": true
  }
}
{
  "recovery": {
    "before": {
      "rows": 75,
      "business_digest": "1321d375742d806a1f0fef82be9e2862af04f5d18f3a976792a6b1d9aa1b4383",
      "version": 0
    },
    "after_delete": {
      "rows": 74,
      "business_digest": "69ba5ff6a74a120af5b9144d927c1916735fdaf2eaaba2168accc9b1182a2def",
      "version": 1
    },
    "after_restore": {
      "rows": 75,
      "business_digest": "1321d375742d806a1f0fef82be9e2862af04f5d18f3a976792a6b1d9aa1b4383",
      "version": 2
    },
    "history": [
      {"version": 2, "operation": "RESTORE", "engineInfo": "Apache-Spark/3.5.8 Delta-Lake/3.3.3"},
      {"version": 1, "operation"

In [156]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day03_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day03_handoff.zip
